In [ ]:
# Import section
from backtesting import Backtest
import pandas as pd
from backtesting import Strategy
from backtesting.lib import crossover
import numpy as np
import math

# META
MARGIN = 0.01
CASH = 10_000
COMMISSION = 0.00002
table_name = 'f1_session'

SESSIONS = [
    {'name': 'ASIAN', 'start': '03:00:00', 'end': '09:00:00'},
    {'name': 'LONDON', 'start': '10:30:00', 'end': '18:30:00'},
    {'name': 'NEWYORK', 'start': '16:30:00', 'end': '23:00:00'},
    {'name': 'OVERLAP', 'start': '16:30:00', 'end': '18:30:00'}
]


# Strategy

In [ ]:
class STRAT(Strategy):
    n1 = 9
    n2 = 21

    risk_pct = 0.01
    risk_reward = 1
    session = 'ASIAN'

    def init(self):
        super().init()
        close = pd.Series(self.data.Close)
        self.ema1 = self.I(lambda: close.ewm(span=self.n1, adjust=False).mean(), name="ema fast")
        self.ema2 = self.I(lambda: close.ewm(span=self.n2, adjust=False).mean(), name="ema slow")

    def _get_session(self):
        for item in sessions:
            if item['name'] == self.session:
                return item
        return None

    def next(self):
        price = self.data.Close[-1]
        prev_low = self.data.Low[-2]
        prev_high = self.data.High[-2]

        session = self._get_session()
        if session is None:
            return

        current_time = self.data.index[-1].time()
        start_time = pd.Timestamp(session['start']).time()
        end_time = pd.Timestamp(session['end']).time()

        if current_time < start_time or current_time >= end_time:
            if self.position:
                self.position.close()
            return

        long_signal = crossover(self.ema1, self.ema2)
        short_signal = crossover(self.ema2, self.ema1)

        risk_amt = max(self.equity, CASH) * self.risk_pct

        def get_size_by_leveraged(sl_dist):
            ideal_size = risk_amt / sl_dist
            max_leverage_size = (self.equity * 0.85) / (price * MARGIN)
            size = int(min(ideal_size, max_leverage_size))
            return size

        if long_signal:
            sl_dist = abs(price - prev_low)
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                return

            size = get_size_by_leveraged(sl_dist)

            if size <= 0:
                return

            self.position.close()
            self.buy(size=size, sl=price - sl_dist, tp=price + tp_dist)

        elif short_signal:
            sl_dist = abs(price - prev_high)
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                return

            size = get_size_by_leveraged(sl_dist)

            if size <= 0:
                return

            self.position.close()
            self.sell(size=size, sl=price + sl_dist, tp=price - tp_dist)


# Basic backtest

In [9]:
# Load sample data and run a basic backtest
file_path = "~/market_data/XAUUSD/XAUUSD_M5.csv"

try:
  # Load data
  data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
  
  # Run backtest
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
  stats = bt.run()
  
  print(stats)
  
except Exception as e:
  print(f"Error: {str(e)}")

Start                     2020-01-02 01:00:00
End                       2025-12-31 23:55:00
Duration                   2190 days 22:55:00
Exposure Time [%]                     1.77439
Equity Final [$]                     18.68298
Equity Peak [$]                   10142.10106
Commissions [$]                    5469.32802
Return [%]                          -99.81317
Buy & Hold Return [%]               184.03997
Return (Ann.) [%]                   -64.44127
Volatility (Ann.) [%]                30.37986
CAGR [%]                            -64.91448
Sharpe Ratio                         -2.12118
Sortino Ratio                        -1.07106
Calmar Ratio                         -0.64553
Alpha [%]                          -102.58917
Beta                                  0.01508
Max. Drawdown [%]                    -99.8271
Avg. Drawdown [%]                   -21.26456
Max. Drawdown Duration     2176 days 19:30:00
Avg. Drawdown Duration      438 days 04:03:00
# Trades                          

# SQLite connection

In [10]:
import sqlite3

# Setup SQLite database
db_path = "../backtest_results.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Database setup complete at {db_path}")

conn.commit()

Database setup complete at ../backtest_results.db


# Combination test

In [11]:
from itertools import product

cursor.execute(f'''
CREATE TABLE IF NOT EXISTS {table_name} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    risk_pct REAL NOT NULL,
    risk_reward REAL NOT NULL,
    session_name TEXT NOT NULL,
    return_pct REAL,
    max_drawdown_pct REAL,
    expectancy REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    win_rate_pct REAL,
    num_trades INTEGER,
    equity_final REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')


assets = ['XAUUSD', 'EURUSD', 'GBPUSD', 'USDCAD', 'AUDUSD', 'USDCHF', 'US100', 'US500']
timeframes = ['M1', 'M5', 'M15']

init_params = {
    'n1': 9,
    'n2': 21,
    'risk_pct': 0.01,
    'risk_reward': 0.5
}

parameters = [
    timeframes,
    assets,
    [0.25, 0.33, 0.5, *range(1, 2)],
    [0.001, 0.005, 0.01],
    SESSIONS
]

total_combinations = math.prod(len(p) for p in parameters)
print(f'Testing {total_combinations} combinations...')

for timeframe, asset, rr, risk_pct, session in product(*parameters):
    file_path = f'~/market_data/{asset}/{asset}_{timeframe}.csv'

    try:
        data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
        bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
        stats = bt.run(risk_reward=rr, risk_pct=risk_pct, session=session['name'])

        return_pct = stats['Return [%]']
        max_drawdown_pct = stats['Max. Drawdown [%]']
        expectancy = stats.get('Expectancy [%]', stats.get('Expectancy', None))
        sharpe_ratio = stats['Sharpe Ratio']
        profit_factor = stats['Profit Factor']
        win_rate_pct = stats['Win Rate [%]']
        num_trades = stats['# Trades']
        equity_final = stats['Equity Final [$]']

        cursor.execute(f'''
            INSERT INTO {table_name}
            (asset, timeframe, n1, n2, risk_pct, risk_reward, session_name, return_pct, max_drawdown_pct, expectancy, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            asset,
            timeframe,
            init_params['n1'],
            init_params['n2'],
            risk_pct,
            rr,
            session['name'],
            return_pct,
            max_drawdown_pct,
            expectancy,
            sharpe_ratio,
            profit_factor,
            win_rate_pct,
            num_trades,
            equity_final
        ))
        conn.commit()

        print(f"✓ {asset}_{timeframe} [{session['name']}]: Return={return_pct:.2f}%, DD={max_drawdown_pct:.2f}%, Sharpe={sharpe_ratio:.2f}, WinRate={win_rate_pct:.2f}%")

    except Exception as e:
        print(f"✗ {asset}_{timeframe} [{session['name']}]: Error - {str(e)}")
        cursor.execute(f'''
            INSERT INTO {table_name}
            (asset, timeframe, n1, n2, risk_pct, risk_reward, session_name, return_pct, max_drawdown_pct, expectancy, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL)
        ''', (
            asset,
            timeframe,
            init_params['n1'],
            init_params['n2'],
            risk_pct,
            rr,
            session['name']
        ))
        conn.commit()

print(f'Results stored in {db_path}')
print('\nBacktesting complete!')

Testing 1152 combinations...
✓ XAUUSD_M1 [ASIAN]: Return=-99.82%, DD=-99.82%, Sharpe=-6.20, WinRate=45.68%
✓ XAUUSD_M1 [LONDON]: Return=-99.81%, DD=-99.81%, Sharpe=-4.92, WinRate=57.47%
✓ XAUUSD_M1 [NEWYORK]: Return=-99.81%, DD=-99.83%, Sharpe=-5.78, WinRate=54.97%
✓ XAUUSD_M1 [OVERLAP]: Return=-99.78%, DD=-99.78%, Sharpe=-6.88, WinRate=66.11%
✓ XAUUSD_M1 [ASIAN]: Return=-99.82%, DD=-99.82%, Sharpe=-3.77, WinRate=43.92%
✓ XAUUSD_M1 [LONDON]: Return=-99.86%, DD=-99.86%, Sharpe=-3.16, WinRate=57.62%
✓ XAUUSD_M1 [NEWYORK]: Return=-99.82%, DD=-99.82%, Sharpe=-3.36, WinRate=56.49%
✓ XAUUSD_M1 [OVERLAP]: Return=-99.82%, DD=-99.82%, Sharpe=-4.03, WinRate=64.04%
✓ XAUUSD_M1 [ASIAN]: Return=-99.81%, DD=-99.81%, Sharpe=-3.89, WinRate=44.70%
✓ XAUUSD_M1 [LONDON]: Return=-99.84%, DD=-99.84%, Sharpe=-3.01, WinRate=57.45%
✓ XAUUSD_M1 [NEWYORK]: Return=-99.88%, DD=-99.88%, Sharpe=-3.36, WinRate=55.93%
✓ XAUUSD_M1 [OVERLAP]: Return=-99.81%, DD=-99.81%, Sharpe=-3.82, WinRate=62.79%
✓ XAUUSD_M1 [ASIAN]:

KeyboardInterrupt: 

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

query = f"""
SELECT max_drawdown_pct, win_rate_pct
FROM {table_name}
WHERE max_drawdown_pct IS NOT NULL AND win_rate_pct IS NOT NULL and num_trades >= 1000
"""
df_stats = pd.read_sql_query(query, conn)

mean_dd = df_stats['max_drawdown_pct'].mean()
mean_wr = df_stats['win_rate_pct'].mean()

print(f"Mean max drawdown (%): {mean_dd:.4f}")
print(f"Mean win rate (%): {mean_wr:.4f}")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Distribution of Max Drawdown (%)", "Distribution of Win Rate (%)"]
)

fig.add_trace(go.Histogram(x=df_stats["max_drawdown_pct"], name="Max Drawdown", nbinsx=40), row=1, col=1)
fig.add_trace(go.Histogram(x=df_stats["win_rate_pct"], name="Win Rate", nbinsx=40), row=1, col=2)

fig.update_layout(
    title="Interactive Backtest Distributions",
    bargap=0.1,
    template="plotly_white"
)
fig.show()

In [ ]:
bt.plot(open_browser=False, filename=f"{table_name}.html", resample=False, plot_pl=False, plot_volume=False)